# Causal Feature Engineering

Reproducible feature-engineering record for AeroXAI.

This notebook replaces the orchestration in `ml/data/build_features.py`.
Reusable causal transformations remain in `ml/data/features.py`, and schema normalization remains in `ml/data/schema.py`.

It builds trailing 5-minute features, applies the telemetry-coverage rule, creates chronological partitions, writes processed Parquet files, and regenerates `docs/preprocessing_report.json`.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()

if not (ROOT / "ml").exists() and (ROOT.parent / "ml").exists():
    ROOT = ROOT.parent

if not (ROOT / "ml").exists():
    raise RuntimeError(
        "Could not locate repository root. "
        "Open this notebook from the xAI-Compressor repository."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Repository root: {ROOT}")

Repository root: C:\Users\Quoc Thai\Downloads\AI\xAI-Compressor


In [2]:
import json

import pandas as pd
import yaml

from ml.data.features import (
    QUALITY_COLUMNS,
    aggregate_causal_bins,
    filter_valid_bins,
    model_feature_columns,
    select_time_range,
)
from ml.data.schema import normalize_raw_frame

CONFIG_PATH = ROOT / "configs" / "metropt.yaml"
REPORT_PATH = ROOT / "docs" / "preprocessing_report.json"

with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

dataset_config = config["dataset"]
preprocessing = config["preprocessing"]
split_config = config["split"]

In [3]:
raw_path = ROOT / dataset_config["raw_csv"]

raw = pd.read_csv(
    raw_path,
    low_memory=False,
)

frame = normalize_raw_frame(raw)

print(f"Normalized rows: {len(frame):,}")

Normalized rows: 1,516,948


## Aggregate causal 5-minute features

Each feature at timestamp `t` uses only observations at or before `t`. No interpolation, backfill, centered rolling window, or future-looking transform is used.

In [4]:
features = aggregate_causal_bins(
    frame,
    bin_minutes=int(preprocessing["bin_minutes"]),
    expected_samples=int(
        preprocessing["expected_samples_per_bin"]
    ),
)

model_columns = model_feature_columns(
    features.columns
)

print(f"Calendar bins:  {len(features):,}")
print(f"Model features: {len(model_columns)}")
print(f"Quality fields: {QUALITY_COLUMNS}")

Calendar bins:  61,393
Model features: 63
Quality fields: ['sample_count', 'coverage_ratio']


## Quality filter

In [5]:
adequate_coverage = (
    features["coverage_ratio"]
    >= float(preprocessing["minimum_coverage"])
)

covered_features = features.loc[
    adequate_coverage,
    model_columns,
]

complete_after_coverage = (
    covered_features
    .notna()
    .all(axis=1)
)

clean = filter_valid_bins(
    features,
    minimum_coverage=float(
        preprocessing["minimum_coverage"]
    ),
)

low_coverage_bins = int(
    (~adequate_coverage).sum()
)

incomplete_bins = int(
    adequate_coverage.sum()
    - complete_after_coverage.sum()
)

print(f"Valid bins:              {len(clean):,}")
print(f"Low-coverage bins:       {low_coverage_bins:,}")
print(f"Incomplete covered bins: {incomplete_bins:,}")

Valid bins:              50,301
Low-coverage bins:       11,092
Incomplete covered bins: 0


## Chronological train / calibration / test partitions

In [6]:
train = select_time_range(
    clean,
    **split_config["train"],
)

calibration = select_time_range(
    clean,
    **split_config["calibration"],
)

test = select_time_range(
    clean,
    **split_config["test"],
)

def frame_summary(data: pd.DataFrame) -> dict:
    if data.empty:
        return {
            "rows": 0,
            "start": None,
            "end": None,
        }

    return {
        "rows": len(data),
        "start": str(data.index.min()),
        "end": str(data.index.max()),
    }

pd.DataFrame(
    {
        "train": frame_summary(train),
        "calibration": frame_summary(calibration),
        "test": frame_summary(test),
    }
).T

,rows,start,end
train,5271,2020-02-01 00:05:00,2020-02-21 23:55:00
calibration,1796,2020-02-22 01:00:00,2020-02-28 23:55:00
test,43221,2020-03-01 04:05:00,2020-09-01 03:55:00


## Persist processed datasets

These Parquet outputs remain local/ignored artifacts and can be regenerated from the frozen raw data and configuration.

In [7]:
outputs = {
    "features_file": clean,
    "train_file": train,
    "calibration_file": calibration,
    "test_file": test,
}

for key, output_frame in outputs.items():
    target = ROOT / preprocessing[key]
    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    output_frame.to_parquet(
        target,
        engine="pyarrow",
    )
    print(
        f"{key}: {len(output_frame):,} rows -> {target}"
    )

features_file: 50,301 rows -> C:\Users\Quoc Thai\Downloads\AI\xAI-Compressor\data\processed\features.parquet
train_file: 5,271 rows -> C:\Users\Quoc Thai\Downloads\AI\xAI-Compressor\data\processed\train.parquet
calibration_file: 1,796 rows -> C:\Users\Quoc Thai\Downloads\AI\xAI-Compressor\data\processed\calibration.parquet
test_file: 43,221 rows -> C:\Users\Quoc Thai\Downloads\AI\xAI-Compressor\data\processed\test.parquet


## Incident coverage after filtering

In [8]:
incident_coverage = []
bin_minutes = int(
    preprocessing["bin_minutes"]
)

for incident in dataset_config["reported_incidents"]:
    start = pd.Timestamp(incident["start"])
    end = pd.Timestamp(incident["end"])

    valid_mask = (
        (clean.index >= start)
        & (clean.index <= end)
    )

    expected_bins = int(
        (
            (end - start)
            / pd.Timedelta(minutes=bin_minutes)
        )
        + 1
    )

    valid_bins = int(valid_mask.sum())

    incident_coverage.append(
        {
            "id": incident["id"],
            "expected_bins": expected_bins,
            "valid_bins": valid_bins,
            "coverage_ratio": (
                valid_bins / expected_bins
                if expected_bins
                else None
            ),
        }
    )

pd.DataFrame(incident_coverage)

C:\Users\Quoc Thai\AppData\Local\Temp\ipykernel_24768\3926807983.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  / pd.Timedelta(minutes=bin_minutes)


,id,expected_bins,valid_bins,coverage_ratio
0,1,288,286,0.993056
1,2,79,79,1.000000
2,3,631,572,0.906498
3,4,55,54,0.981818


## Write preprocessing report

In [9]:
coverage_quantiles = (
    features["coverage_ratio"]
    .quantile(
        [
            0.00,
            0.01,
            0.05,
            0.50,
            0.95,
            0.99,
            1.00,
        ]
    )
    .to_dict()
)

report = {
    "raw_rows": len(frame),
    "bins_before_quality_filter": len(features),
    "bins_after_quality_filter": len(clean),
    "low_coverage_bins": low_coverage_bins,
    "incomplete_feature_bins": incomplete_bins,
    "model_feature_count": len(model_columns),
    "quality_columns": QUALITY_COLUMNS,
    "coverage_quantiles": {
        str(key): float(value)
        for key, value
        in coverage_quantiles.items()
    },
    "splits": {
        "train": frame_summary(train),
        "calibration": frame_summary(calibration),
        "test": frame_summary(test),
    },
    "incident_coverage": incident_coverage,
}

REPORT_PATH.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

if not (
    train.index.max() < calibration.index.min()
    and calibration.index.max() < test.index.min()
):
    raise ValueError(
        "Train, calibration, and test partitions "
        "must be strictly chronological and non-overlapping."
    )

print(f"Preprocessing report written to: {REPORT_PATH}")

Preprocessing report written to: C:\Users\Quoc Thai\Downloads\AI\xAI-Compressor\docs\preprocessing_report.json
